# SFN-World v0.4 — not yet learned vs. unlearnable
Run top to bottom on a RunPod A100 (PyTorch template). Every run is resumable: finished runs in `OUT` are skipped,
so after an interruption just re-run the cell. Stage order and go/no-go rules follow §8–§9 of the draft;
cells that depend on an earlier result compute their inputs from it.

In [ ]:
%pip install -q -e . pandas matplotlib
import torch
assert torch.cuda.is_available(), "no GPU visible"
print(torch.cuda.get_device_name(0))

In [ ]:
%matplotlib inline
from IPython.display import display
from sfn.stages import *
from sfn import plots

OUT = "/workspace/results"  # on the RunPod network volume so it survives pod restarts

## Stage 0b — learning time on ground truth (no encoder, 9 runs of seconds)

In [ ]:
execute_gt(OUT)
gt = gt_table(OUT)
display(gt.unstack("diff"))
DIFF = default_difficulty(gt)  # §8 checks 1 and 6: lowest difficulty with t90(5) >= 3 t90(4)
print("default difficulty:", DIFF)

Checks 2–5 of §8 by eye: group 6 plateaus near 0.5, group 3 near 0.90, `r2_lin` within ±0.05 of the §3 table
(0.96, 0.49, 0.90, 1.0, 0.41, 0.21, 0.93 at D0).
## Stage 0 — reproduce the failure (12 runs) and calibrate M2's β

In [ ]:
execute(stage0(DIFF), OUT)
df = collect(OUT)
print(f"mean {df.seconds.mean():.0f}s per run -> full budget ~ {df.seconds.mean() * 117 / 3600:.1f} h")
display(table(df, arm="A0", diff=DIFF, heads=True).T)
s = failing_setting(df)  # raises if A0 never drops group 5 while keeping group 1
LAM, W_FAST, BETA = s["lam"], s["w_fast"], s["beta"]
print(s)
table_, ok = premise_check(df, arm="A0", diff=DIFF, heads=True, lam=LAM, w_fast=W_FAST)
display(table_)
print("PREMISE", "holds" if ok else "FAILS: slow groups do not carry more variance than fast-hard ones; "
      "A0 keeping group 5 would be a SIGReg-budget effect, so do not read Stage 1 as a gating test")

## Stage 1 — main comparison (13 target arms × 3 seeds, all B0, D = 16)

In [ ]:
common = dict(lam=LAM, w_fast=W_FAST, diff=DIFF, beta=BETA)
view = dict(lam=LAM, w_fast=W_FAST, heads=False)
execute(stage1(**common), OUT)
df = collect(OUT)
t = table(df, D=16, diff=DIFF, bfac="B0", **view)
display(t)
display(gate_verdict(t, [a for a in t.index if a != "A0"]))

In [ ]:
plots.race(df, OUT, ("A0", "A2", "M1d"), D=16, diff=DIFF, **view)
plots.calibration(t)
plots.pareto(df, D=16, diff=DIFF, bfac="B0", **view);

## Stage 1-B — SIGReg budget factor (12 runs)

In [ ]:
execute(stage1_budget(**common), OUT)
df = collect(OUT)
t = table(df, D=16, diff=DIFF, **view)
display(t.loc[[a for a in t.index if a.startswith(("A0", "M1d"))]])
display(budget_verdict(t, "M1d", "B1"), budget_verdict(t, "M1d", "B2"))

## Stage 1b — capacity D = 48 (15 runs)

In [ ]:
best_m = best_cfg(df, M_ARMS, D=16, diff=DIFF, bfac="B0", **view)["arm"]
print("best M arm:", best_m)
execute(stage1_capacity(best_m, **common), OUT)
df = collect(OUT)
display(table(df, D=48, diff=DIFF, **view))
plots.budget(df, diff=DIFF, **view);

## Stage 1c — D = 24 (9 runs, conditional)
Runs only if factor B shows no effect at D = 16 (§8).

In [ ]:
t16 = table(df, D=16, diff=DIFF, **view)
b_effect = budget_verdict(t16)["B beats G by >= 0.1 on R2_5/R2_6"]
if not b_effect:
    execute(stage1_middle(**common), OUT)
    df = collect(OUT)
    display(table(df, D=24, diff=DIFF, **view))
else:
    print("B has an effect at D = 16: Stage 1c skipped")

## Stage 2 — difficulty sweep (30 runs)

In [ ]:
best_a1 = best_cfg(df, ["A1"], D=16, diff=DIFF, bfac="B0", **view)
best_g = best_cfg(df, ["M1b", "M1d"], D=16, diff=DIFF, bfac="B0", **view)
print(best_a1, best_g)
execute(stage2(best_a1, best_g, **common), OUT)
df = collect(OUT)
for d in (1, 2):
    t = table(df, D=16, diff=d, bfac="B0", **view)
    display(t)
    display(gate_verdict(t, [a for a in t.index if a != "A0"]))

## Stage 2-B — best G+B combination at D1, D2 (6 runs, conditional)
Runs only if factor B passed its §9 criterion (at D = 24 instead if Stage 1c ran).

In [ ]:
D_B = 24 if not b_effect else 16
tb = table(df, D=D_B, diff=DIFF, **view)
if all(budget_verdict(tb, "M1d", "B1", rand_tab=t16).values()):  # 1c has no B-rand arm
    execute(stage2_budget(best_cfg(df, ["M1d"], D=D_B, diff=DIFF, **view), **common), OUT)
    df = collect(OUT)
    display(table(df, D=16, diff=1, **view), table(df, D=16, diff=2, **view))
else:
    print("factor B did not pass its criterion: Stage 2-B skipped")

## All runs

In [ ]:
df = collect(OUT)
df.to_csv(f"{OUT}/summary.csv", index=False)
print(len(df), "runs;", f"{df.seconds.sum() / 3600:.1f} GPU-hours")